## Objective:
To decompose the dense empirical covariance matrix ($\Sigma$) into strictly independent, orthogonal dimensions of risk. We must mathematically isolate the macro market forces from the micro sector relationships, calculate the absolute kinetic energy contained within each force, and mathematically sever the dimensions that represent pure statistical noise.

**The Physics Analogy: The Football & The Steel Cage** 

If you plot the returns of highly correlated assets in a multi-dimensional scatter plot, the data forms the shape of a tilted football. Standard $X, Y, Z$ axes (individual asset returns) smear the risk across multiple dimensions. The Eigenbasis is a mathematical steel cage designed to perfectly align with the football. The primary eigenvector ($w_1$) drives a steel bar straight through the longest axis of the football, capturing the maximum possible mass (The Macro Tide). Every subsequent vector ($w_2, w_3$) is driven through the remaining axes at strict 90-degree angles, ensuring the forces never overlap.

**Mathematical Execution:**

Because our covariance matrix ($\Sigma \in \mathbb{R}^{N \times N}$) is symmetric and Positive Semi-Definite, the Spectral Theorem guarantees it can be perfectly diagonalized.

1. **Hermitian Eigendecomposition:** $$\Sigma = W \Lambda W^T$$ 
- $W$: The matrix of Eigenvectors. These are orthogonal coordinates. $W^T W = I$.
- $\Lambda$: The diagonal matrix of Eigenvalues ($\lambda_i$). This dictates the exact magnitude of kinetic energy within each vector.

2. **Trace Equivalence (Variance Attribution):** We translate raw eigenvalue magnitude into a percentage of Total System Risk.$$V_i = \frac{\lambda_i}{\sum_{j=1}^N \lambda_j}$$

3. **Dimensional Severance:** We compute the cumulative sum of $V_i$ and drop all vectors $\ge$ a defined threshold (e.g., $0.95$).

In [26]:
import sys
import numpy as np
from pathlib import Path

current_dir = Path.cwd()
root_dir = current_dir.parent
sys.path.append(str(root_dir))

from src.covariance import empirical_cov_matrix

path = data_path = root_dir / 'data' / 'synthetic_market_19.npy'
data = np.load(path)

sigma = empirical_cov_matrix(data)

## I. Hermitian decomposition

In [27]:
def extract_eigen(sigma: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    eigenValues, eigenVectors = np.linalg.eigh(sigma)
    
    sorted_eigenvalues_index = np.argsort(eigenValues)[::-1]
    
    sorted_eigenvectors = eigenVectors[:, sorted_eigenvalues_index]
    sorted_eigenvalues = eigenValues[sorted_eigenvalues_index]
    
    return (sorted_eigenvalues, sorted_eigenvectors)

eigenVal, eigenVec = extract_eigen(sigma)
eigenVal, eigenVec

(array([4.33676301, 0.48614478, 0.43638901, 0.36285887, 0.15940347]),
 array([[-0.29604521, -0.02620016, -0.02165459,  0.05891619, -0.95274905],
        [-0.65912267,  0.34921353, -0.45801319, -0.44968428,  0.17780659],
        [-0.40252431,  0.07269032,  0.88194677, -0.21638633,  0.0896501 ],
        [-0.35047857, -0.91987696, -0.1072038 , -0.0383633 ,  0.13426375],
        [-0.43937852,  0.16095329, -0.02079021,  0.86372342,  0.18598434]]))

## II. Trace equivalence

In [28]:
def variance_explained(eigenvalues: np.ndarray) -> np.ndarray:
    eigenvalues = eigenvalues / np.sum(eigenvalues)
    return eigenvalues

variance = variance_explained(eigenVal)
variance

array([0.75010268, 0.08408541, 0.07547947, 0.06276142, 0.02757102])

## III. Dimension reductions

In [29]:
def reduce_dimensions(eigenvalues: np.ndarray, eigenvectors: np.ndarray, threshold=0.95) -> tuple[np.ndarray, np.ndarray]:
    cumulative_variance = np.cumsum(eigenvalues)
    
    cutoff_index = np.argmax(cumulative_variance >= threshold)
    
    compressed_eigenvalues = eigenvalues[:cutoff_index+1]
    compressed_eigenvectors = eigenvectors[:, :cutoff_index+1]
    
    return (compressed_eigenvalues, compressed_eigenvectors)

reduce_dimensions(variance, eigenVec)

(array([0.75010268, 0.08408541, 0.07547947, 0.06276142]),
 array([[-0.29604521, -0.02620016, -0.02165459,  0.05891619],
        [-0.65912267,  0.34921353, -0.45801319, -0.44968428],
        [-0.40252431,  0.07269032,  0.88194677, -0.21638633],
        [-0.35047857, -0.91987696, -0.1072038 , -0.0383633 ],
        [-0.43937852,  0.16095329, -0.02079021,  0.86372342]]))